# 06 — Generalization Mitigation

## Objective

The baseline model shows substantial performance degradation on unseen
validation groups, with strongly group-structured errors.

The previous diagnosis did not identify a single measured covariate capable
of explaining the generalization failure. Instead, the results were consistent
with a multifactorial group-level distribution shift.

This notebook evaluates mitigation strategies under the same group-based
validation protocol.

Each experiment is compared against the original baseline using both
image-level and group-level metrics.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = Path("/content/brain-tumor-mri")

print("TensorFlow:", tf.__version__)
print("Project root exists:", PROJECT_ROOT.exists())

TensorFlow: 2.20.0
Project root exists: False


In [3]:
import os

%cd /content

if not os.path.exists("/content/brain-tumor-mri"):
    !git clone https://github.com/nouraastafofana-droid/brain-tumor-mri.git

%cd /content/brain-tumor-mri

PROJECT_ROOT = Path("/content/brain-tumor-mri")

print("Project root exists:", PROJECT_ROOT.exists())
print("Current directory:", Path.cwd())

/content
Cloning into 'brain-tumor-mri'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 53 (delta 21), reused 23 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 9.51 MiB | 27.19 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/brain-tumor-mri
Project root exists: True
Current directory: /content/brain-tumor-mri


In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
from pathlib import Path

MODEL_DIR = Path("/content/drive/MyDrive/brain-tumor-mri/models")

for p in MODEL_DIR.iterdir():
    print(p.name, "->", round(p.stat().st_size / 1024**2, 2), "MB")

efficientnetv2b0_baseline.keras -> 23.65 MB
efficientnetv2b0_finetuned.keras -> 32.45 MB
efficientnetv2b0_300_baseline.keras -> 23.65 MB
swin_tiny_224_baseline.pt -> 105.14 MB
swin_tiny_224_finetuned.pt -> 159.27 MB


In [7]:
BASELINE_MODEL_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

In [8]:
BASELINE = {
    "image_accuracy": 0.3013537374926427,
    "macro_group_accuracy": 0.2748687640590233,
    "median_group_accuracy": 0.15384615384615385,
    "zero_accuracy_groups": 24,
    "n_groups": 55,
}

pd.Series(BASELINE)

,0
image_accuracy,0.301354
macro_group_accuracy,0.274869
median_group_accuracy,0.153846
zero_accuracy_groups,24.000000
n_groups,55.000000


In [9]:
import tensorflow as tf

BASELINE_MODEL_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

model = tf.keras.models.load_model(BASELINE_MODEL_PATH)

print("Loaded:", BASELINE_MODEL_PATH)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters:", f"{model.count_params():,}")

Loaded: /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_baseline.keras
Input shape: (None, 224, 224, 3)
Output shape: (None, 10)
Parameters: 5,932,122


In [10]:
import json
from pathlib import Path

NB05_PATH = Path(
    "/content/brain-tumor-mri/notebooks/05_generalization_diagnosis.ipynb"
)

with open(NB05_PATH, "r", encoding="utf-8") as f:
    nb05 = json.load(f)

keywords = [
    "train_df",
    "val_df",
    "metadata",
    "split_group",
    "read_csv",
    "read_parquet",
]

for i, cell in enumerate(nb05["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword in source for keyword in keywords):
        print("\n" + "=" * 80)
        print(f"CELL {i}")
        print("=" * 80)
        print(source[:6000])


CELL 5
import shutil

DRIVE_MANIFEST = Path(
    "/content/drive/MyDrive/classification_manifest.csv"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Frozen manifest on Drive:",
    DRIVE_MANIFEST.exists()
)

assert DRIVE_MANIFEST.exists(), (
    "Frozen classification manifest not found on Google Drive."
)

shutil.copy2(
    DRIVE_MANIFEST,
    MANIFEST_PATH
)

manifest = pd.read_csv(
    MANIFEST_PATH
)

print(
    "Manifest shape:",
    manifest.shape
)

print()
print(
    manifest["split"].value_counts()
)

CELL 6
manifest = pd.read_csv(MANIFEST_PATH)

print("Manifest shape:", manifest.shape)
print(manifest.columns.tolist())

CELL 7
# Recover frozen partitions
train_df = manifest[
    manifest["split"] == "train"
].copy()

val_df = manifest[
    manifest["split"] == "validation"
].copy()

test_df = manifest[
    manifest["split"] == "test"
].copy()

print("Images")
print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test

In [11]:
import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path(
    "/content/drive/MyDrive/classification_manifest.csv"
)

assert MANIFEST_PATH.exists(), (
    f"Manifest not found: {MANIFEST_PATH}"
)

manifest = pd.read_csv(MANIFEST_PATH)

train_df = manifest[
    manifest["split"] == "train"
].copy()

val_df = manifest[
    manifest["split"] == "validation"
].copy()

test_df = manifest[
    manifest["split"] == "test"
].copy()

print("Manifest:", manifest.shape)

print("\nImages")
print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test:       {len(test_df):,}")

print("\nGroups")
print(f"Train:      {train_df['split_group'].nunique()}")
print(f"Validation: {val_df['split_group'].nunique()}")
print(f"Test:       {test_df['split_group'].nunique()}")

# Safety checks
assert set(train_df["split_group"]).isdisjoint(
    val_df["split_group"]
)

assert set(train_df["split_group"]).isdisjoint(
    test_df["split_group"]
)

assert set(val_df["split_group"]).isdisjoint(
    test_df["split_group"]
)

print("\n✓ Frozen partitions recovered")
print("✓ No split_group leakage")

Manifest: (11300, 12)

Images
Train:      7,928
Validation: 1,699
Test:       1,673

Groups
Train:      247
Validation: 55
Test:       55

✓ Frozen partitions recovered
✓ No split_group leakage


In [12]:
PROJECT_ROOT / val_df["resolved_relative_path"]

,resolved_relative_path
9601,/content/brain-tumor-mri/data/raw/Astrocytoma ...
9602,/content/brain-tumor-mri/data/raw/Astrocytoma ...
9603,/content/brain-tumor-mri/data/raw/Astrocytoma ...
9604,/content/brain-tumor-mri/data/raw/Astrocytoma ...
9605,/content/brain-tumor-mri/data/raw/Astrocytoma ...
...,...
11295,/content/brain-tumor-mri/data/raw/Schwannoma T...
11296,/content/brain-tumor-mri/data/raw/Schwannoma T...
11297,/content/brain-tumor-mri/data/raw/Schwannoma T...
11298,/content/brain-tumor-mri/data/raw/Schwannoma T...


In [16]:
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get(
    "KAGGLE_API_TOKEN"
)

print("✓ Kaggle token loaded.")

✓ Kaggle token loaded.


In [17]:
!kaggle datasets download \
    -d fernando2rad/brain-tumor-mri-images-30-classes \
    -p /content/brain-tumor-mri/data/raw \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/fernando2rad/brain-tumor-mri-images-30-classes
License(s): apache-2.0
100% 478M/478M [00:06<00:00, 76.8MB/s]



In [18]:
from pathlib import Path

DATA_DIR = Path("/content/brain-tumor-mri/data/raw")

print("DATA_DIR exists:", DATA_DIR.exists())
print("JPG files:", len(list(DATA_DIR.rglob("*.jpg"))))
print("DATA.json exists:", (DATA_DIR / "DATA.json").exists())

for split_name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    paths = df["resolved_relative_path"].apply(
        lambda p: PROJECT_ROOT / p
    )

    n_existing = paths.apply(lambda p: p.exists()).sum()

    print(
        f"{split_name:10s}: "
        f"{n_existing:,} / {len(df):,}"
    )

DATA_DIR exists: True
JPG files: 11300
DATA.json exists: True
train     : 7,928 / 7,928
validation: 1,699 / 1,699
test      : 1,673 / 1,673


In [24]:
# ============================================================
# Mitigation strategy 1 — MRI-safe data augmentation
# ============================================================

import tensorflow as tf

mri_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.03,
            fill_mode="constant",
            fill_value=0.0,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="constant",
            fill_value=0.0,
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.05, 0.05),
            width_factor=(-0.05, 0.05),
            fill_mode="constant",
            fill_value=0.0,
        ),
    ],
    name="mri_augmentation",
)

print("✓ MRI-safe augmentation pipeline created")

✓ MRI-safe augmentation pipeline created


In [25]:
print("train_df shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

print("\nExample:")
display(train_df.head(3))

train_df shape: (7928, 12)

Columns:
['relative_path', 'resolved_relative_path', 'pathology', 'modality', 'split_group', 'split', 'width', 'height', 'point', 'location', 'description', 'pixel_hash']

Example:


,relative_path,resolved_relative_path,pathology,modality,split_group,split,width,height,point,location,description,pixel_hash
1673,Astrocytoma T1C+\T1C+ - Anaplastic astrocytoma...,data/raw/Astrocytoma T1C+/T1C+ - Anaplastic as...,Astrocytoma,T1C+,1,train,512,512,"{'x': 280, 'y': 164}","['frontal', 'ventricle']","There is a large, ill-defined region of white ...",1f8a6534647477a4e6213c1ee564c45fbd6f2ef7cbfe98...
1674,Astrocytoma T1C+\T1C+ - Anaplastic astrocytoma...,data/raw/Astrocytoma T1C+/T1C+ - Anaplastic as...,Astrocytoma,T1C+,1,train,512,512,"{'x': 283, 'y': 168}","['frontal', 'ventricle']","There is a large, ill-defined region of white ...",a508d97db64027e0212864ec6b2e0c554a736023372cf5...
1675,Astrocytoma T1C+\T1C+ - Anaplastic astrocytoma...,data/raw/Astrocytoma T1C+/T1C+ - Anaplastic as...,Astrocytoma,T1C+,1,train,512,512,"{'x': 280, 'y': 166}","['frontal', 'ventricle']","There is a large, ill-defined region of white ...",9a520bafdbfb6b7e0c63924cb3f9c86e46c83e5e299c21...


## Build TensorFlow datasets from frozen partitions

Training data receives MRI-safe augmentation.

Validation and test data remain unaugmented so that evaluation
continues to measure generalization on the original frozen partitions.

In [26]:
import numpy as np
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Keep one deterministic class mapping for every partition
CLASS_NAMES = sorted(train_df["pathology"].unique().tolist())
class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print("Classes:", CLASS_NAMES)
print("Number of classes:", len(CLASS_NAMES))
print("Mapping:", class_to_idx)

assert len(CLASS_NAMES) == model.output_shape[-1], (
    f"Dataset has {len(CLASS_NAMES)} classes but model outputs "
    f"{model.output_shape[-1]} classes."
)

Classes: ['Astrocytoma', 'Ependymoma', 'Glioma', 'Hemangiopericytoma', 'Meningioma', 'Neurocytoma', 'Normal', 'Oligodendroglioma', 'Other', 'Schwannoma']
Number of classes: 10
Mapping: {'Astrocytoma': 0, 'Ependymoma': 1, 'Glioma': 2, 'Hemangiopericytoma': 3, 'Meningioma': 4, 'Neurocytoma': 5, 'Normal': 6, 'Oligodendroglioma': 7, 'Other': 8, 'Schwannoma': 9}


In [35]:
def make_absolute_paths(df):
    return [
        str(PROJECT_ROOT / path)
        for path in df["resolved_relative_path"].tolist()
    ]


def load_mri(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)

    image = tf.image.resize(
        image,
        IMG_SIZE,
        method="bilinear"
    )

    image = tf.cast(image, tf.float32)

    return image, label


def build_dataset(df, training=False):
    paths = make_absolute_paths(df)

    labels = np.array(
        [class_to_idx[x] for x in df["pathology"]],
        dtype=np.int32
    )

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_mri,
        num_parallel_calls=AUTOTUNE
    )

    # No external augmentation here.
    # The saved baseline already contains its own
    # MRI-safe data_augmentation layer.

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = build_dataset(train_df, training=True)
val_ds = build_dataset(val_df, training=False)
test_ds = build_dataset(test_df, training=False)

print("✓ Datasets rebuilt without external augmentation")
print("Train:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test:", tf.data.experimental.cardinality(test_ds).numpy())

✓ Datasets rebuilt without external augmentation
Train: 248
Validation: 54
Test: 53


In [30]:
# ============================================================
# Mitigation experiment 1
# Controlled augmentation + conservative fine-tuning
# ============================================================

MITIGATION_MODEL_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

mitigation_model = tf.keras.models.load_model(
    MITIGATION_MODEL_PATH
)

print("✓ Fresh baseline loaded")
print("Parameters:", f"{mitigation_model.count_params():,}")

✓ Fresh baseline loaded
Parameters: 5,932,122


In [31]:
for i, layer in enumerate(mitigation_model.layers):
    print(
        f"{i:3d} | "
        f"{layer.name:35s} | "
        f"{layer.__class__.__name__:25s} | "
        f"trainable={layer.trainable}"
    )

  0 | image                               | InputLayer                | trainable=True
  1 | data_augmentation                   | Sequential                | trainable=True
  2 | efficientnetv2-b0                   | Functional                | trainable=False
  3 | global_average_pooling              | GlobalAveragePooling2D    | trainable=True
  4 | dropout                             | Dropout                   | trainable=True
  5 | classifier                          | Dense                     | trainable=True


In [32]:
aug_layer = mitigation_model.get_layer("data_augmentation")

print("Augmentation pipeline:")
for i, layer in enumerate(aug_layer.layers):
    print(
        f"{i:2d} | "
        f"{layer.name:30s} | "
        f"{layer.__class__.__name__:25s} | "
        f"config={layer.get_config()}"
    )

Augmentation pipeline:
 0 | random_rotation                | RandomRotation            | config={'name': 'random_rotation', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'factor': (-0.03, 0.03), 'data_format': 'channels_last', 'fill_mode': 'reflect', 'fill_value': 0.0, 'interpolation': 'bilinear', 'seed': None}
 1 | random_translation             | RandomTranslation         | config={'name': 'random_translation', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'height_factor': 0.03, 'width_factor': 0.03, 'fill_mode': 'reflect', 'interpolation': 'bilinear', 'seed': None, 'fill_value': 0.0, 'data_format': 'channels_last'}
 2 | random_zoom                    | RandomZoom                | config={'name': 'random_zoom', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'floa

In [33]:
backbone = mitigation_model.get_layer("efficientnetv2-b0")

print("Backbone layers:", len(backbone.layers))
print("Backbone parameters:", f"{backbone.count_params():,}")
print("Backbone trainable:", backbone.trainable)

print("\nLast 30 backbone layers:")

for i, layer in list(enumerate(backbone.layers))[-30:]:
    print(
        f"{i:3d} | "
        f"{layer.name:35s} | "
        f"{layer.__class__.__name__:25s} | "
        f"trainable={layer.trainable}"
    )

Backbone layers: 270
Backbone parameters: 5,919,312
Backbone trainable: False

Last 30 backbone layers:
240 | block6g_dwconv2                     | DepthwiseConv2D           | trainable=False
241 | block6g_bn                          | BatchNormalization        | trainable=False
242 | block6g_activation                  | Activation                | trainable=False
243 | block6g_se_squeeze                  | GlobalAveragePooling2D    | trainable=False
244 | block6g_se_reshape                  | Reshape                   | trainable=False
245 | block6g_se_reduce                   | Conv2D                    | trainable=False
246 | block6g_se_expand                   | Conv2D                    | trainable=False
247 | block6g_se_excite                   | Multiply                  | trainable=False
248 | block6g_project_conv                | Conv2D                    | trainable=False
249 | block6g_project_bn                  | BatchNormalization        | trainable=False
250 | block6g_dr

In [36]:
backbone = mitigation_model.get_layer("efficientnetv2-b0")

# Allow individual backbone layers to become trainable
backbone.trainable = True

# Freeze the entire backbone first
for layer in backbone.layers:
    layer.trainable = False

# Unfreeze only the final block (block6h) and top convolution.
# BatchNormalization layers remain frozen.
for layer in backbone.layers:
    if layer.name.startswith("block6h") or layer.name == "top_conv":
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True


# Check exactly what will be trained
trainable_backbone_layers = [
    layer.name
    for layer in backbone.layers
    if layer.trainable
]

print("Trainable backbone layers:")
for name in trainable_backbone_layers:
    print(" -", name)

print("\nNumber of trainable backbone layers:",
      len(trainable_backbone_layers))

print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in mitigation_model.trainable_weights):,}"
)

print(
    "Total parameters:",
    f"{mitigation_model.count_params():,}"
)

bn_trainable = [
    layer.name
    for layer in backbone.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
    and layer.trainable
]

print("\nTrainable BatchNorm layers:", bn_trainable)

Trainable backbone layers:
 - block6h_expand_conv
 - block6h_expand_activation
 - block6h_dwconv2
 - block6h_activation
 - block6h_se_squeeze
 - block6h_se_reshape
 - block6h_se_reduce
 - block6h_se_expand
 - block6h_se_excite
 - block6h_project_conv
 - block6h_drop
 - block6h_add
 - top_conv

Number of trainable backbone layers: 13
Trainable parameters: 823,098
Total parameters: 5,932,122

Trainable BatchNorm layers: []


In [37]:
# ============================================================
# Compile conservative fine-tuning experiment
# ============================================================

FINE_TUNE_LR = 1e-5

mitigation_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINE_TUNE_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

print("✓ Mitigation model compiled")
print("Learning rate:", FINE_TUNE_LR)
print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in mitigation_model.trainable_weights):,}"
)

✓ Mitigation model compiled
Learning rate: 1e-05
Trainable parameters: 823,098


In [38]:
# ============================================================
# Fine-tuning callbacks
# ============================================================

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_mitigation_best.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Fine-tuning callbacks ready")
print("Best model will be saved to:")
print(CHECKPOINT_PATH)

✓ Fine-tuning callbacks ready
Best model will be saved to:
/content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation_best.keras


In [39]:
# ============================================================
# Run mitigation experiment 1
# ============================================================

EPOCHS = 15

history_mitigation = mitigation_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4958 - loss: 1.4813
Epoch 1: val_loss improved from None to 2.11023, saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation_best.keras
248/248 ━━━━━━━━━━━━━━━━━━━━ 46s 113ms/step - accuracy: 0.5040 - loss: 1.4605 - val_accuracy: 0.2896 - val_loss: 2.1102 - learning_rate: 1.0000e-05
Epoch 2/15
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.5195 - loss: 1.4291
Epoch 2: val_loss improved from 2.11023 to 2.09758, saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation_best.keras
248/248 ━━━━━━━━━━━━━━━━━━━━ 32s 96ms/step - accuracy: 0.5148 - loss: 1.4288 - val_accuracy: 0.2961 - val_loss: 2.0976 - learning_rate:

In [40]:
# ============================================================
# Evaluate mitigation experiment 1
# ============================================================

BEST_MITIGATION_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_mitigation_best.keras"
)

best_mitigation_model = tf.keras.models.load_model(
    BEST_MITIGATION_PATH
)

print("✓ Best mitigation checkpoint loaded")

val_loss, val_acc = best_mitigation_model.evaluate(
    val_ds,
    verbose=1
)

test_loss, test_acc = best_mitigation_model.evaluate(
    test_ds,
    verbose=1
)

print("\nMitigation experiment 1")
print(f"Validation accuracy : {val_acc:.4f} ({100*val_acc:.2f}%)")
print(f"Validation loss     : {val_loss:.4f}")
print(f"Test accuracy       : {test_acc:.4f} ({100*test_acc:.2f}%)")
print(f"Test loss           : {test_loss:.4f}")

✓ Best mitigation checkpoint loaded
54/54 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.2961 - loss: 2.0976
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.3156 - loss: 2.0486

Mitigation experiment 1
Validation accuracy : 0.2961 (29.61%)
Validation loss     : 2.0976
Test accuracy       : 0.3156 (31.56%)
Test loss           : 2.0486


In [41]:
# ============================================================
# Validation predictions — mitigation experiment 1
# ============================================================

y_true_val = []
y_pred_val = []

for images, labels in val_ds:
    probs = best_mitigation_model.predict(
        images,
        verbose=0
    )

    preds = np.argmax(probs, axis=1)

    y_true_val.extend(labels.numpy())
    y_pred_val.extend(preds)

y_true_val = np.array(y_true_val)
y_pred_val = np.array(y_pred_val)

print("Predictions:", len(y_pred_val))
print("Labels     :", len(y_true_val))
print(
    "Accuracy   :",
    round((y_true_val == y_pred_val).mean(), 4)
)

Predictions: 1699
Labels     : 1699
Accuracy   : 0.2961


In [42]:
# ============================================================
# Group-level evaluation — mitigation experiment 1
# ============================================================

val_mitigation = val_df.copy()

val_mitigation["y_true"] = y_true_val
val_mitigation["y_pred"] = y_pred_val
val_mitigation["correct"] = (
    val_mitigation["y_true"] == val_mitigation["y_pred"]
).astype(int)

mitigation_group_results = (
    val_mitigation
    .groupby(["pathology", "split_group"])
    .agg(
        n_images=("correct", "size"),
        n_correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

mitigation_group_results["accuracy_pct"] = (
    100 * mitigation_group_results["accuracy"]
)

# Global group-level metrics
mitigation_group_summary = {
    "image_accuracy": val_mitigation["correct"].mean(),
    "macro_group_accuracy": mitigation_group_results["accuracy"].mean(),
    "median_group_accuracy": mitigation_group_results["accuracy"].median(),
    "zero_accuracy_groups": int(
        (mitigation_group_results["accuracy"] == 0).sum()
    ),
    "n_groups": len(mitigation_group_results)
}

pd.Series(mitigation_group_summary)

,0
image_accuracy,0.296057
macro_group_accuracy,0.270659
median_group_accuracy,0.111111
zero_accuracy_groups,25.000000
n_groups,55.000000


In [43]:
# ============================================================
# Baseline vs mitigation — group-level comparison
# ============================================================

baseline_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

baseline_probs = baseline_model.predict(
    val_ds,
    verbose=1
)

baseline_pred = np.argmax(
    baseline_probs,
    axis=1
)

baseline_val = val_df.copy()
baseline_val["correct"] = (
    baseline_pred == y_true_val
).astype(int)

baseline_groups = (
    baseline_val
    .groupby(["pathology", "split_group"])
    .agg(
        baseline_accuracy=("correct", "mean")
    )
    .reset_index()
)

comparison_groups = (
    mitigation_group_results
    .merge(
        baseline_groups,
        on=["pathology", "split_group"],
        how="left"
    )
)

comparison_groups["delta"] = (
    comparison_groups["accuracy"]
    - comparison_groups["baseline_accuracy"]
)

comparison_groups["baseline_accuracy_pct"] = (
    100 * comparison_groups["baseline_accuracy"]
)

comparison_groups["mitigation_accuracy_pct"] = (
    100 * comparison_groups["accuracy"]
)

comparison_groups["delta_pct"] = (
    100 * comparison_groups["delta"]
)

comparison_groups[
    [
        "pathology",
        "split_group",
        "n_images",
        "baseline_accuracy_pct",
        "mitigation_accuracy_pct",
        "delta_pct"
    ]
].sort_values("delta_pct")

54/54 ━━━━━━━━━━━━━━━━━━━━ 8s 100ms/step


,pathology,split_group,n_images,baseline_accuracy_pct,mitigation_accuracy_pct,delta_pct
28,Meningioma,151,13,76.923077,46.153846,-30.769231
11,Ependymoma,61,93,60.215054,39.784946,-20.430108
30,Meningioma,154,30,90.000000,70.000000,-20.000000
10,Ependymoma,60,50,24.000000,6.000000,-18.000000
31,Meningioma,156,99,16.161616,0.000000,-16.161616
9,Ependymoma,57,13,38.461538,23.076923,-15.384615
27,Meningioma,150,29,58.620690,44.827586,-13.793103
29,Meningioma,153,37,45.945946,32.432432,-13.513514
22,Meningioma,142,18,100.000000,88.888889,-11.111111
53,Schwannoma,326,36,50.000000,38.888889,-11.111111


In [44]:
print(
    "Groups improved:",
    (comparison_groups["delta"] > 0).sum()
)

print(
    "Groups unchanged:",
    (comparison_groups["delta"] == 0).sum()
)

print(
    "Groups degraded:",
    (comparison_groups["delta"] < 0).sum()
)

print(
    "Baseline zero groups:",
    (comparison_groups["baseline_accuracy"] == 0).sum()
)

print(
    "Mitigation zero groups:",
    (comparison_groups["accuracy"] == 0).sum()
)

rescued = comparison_groups[
    (comparison_groups["baseline_accuracy"] == 0)
    & (comparison_groups["accuracy"] > 0)
]

collapsed = comparison_groups[
    (comparison_groups["baseline_accuracy"] > 0)
    & (comparison_groups["accuracy"] == 0)
]

print("\nZero-accuracy groups rescued:", len(rescued))
print("Groups newly collapsed to zero:", len(collapsed))

Groups improved: 10
Groups unchanged: 32
Groups degraded: 13
Baseline zero groups: 24
Mitigation zero groups: 25

Zero-accuracy groups rescued: 1
Groups newly collapsed to zero: 2


### Mitigation experiment 1 — Conclusion

Conservative fine-tuning of the final EfficientNetV2B0 block did not
improve group-level generalization.

Compared with the frozen baseline:

- Image-level validation accuracy: 30.14% → 29.61%
- Macro group accuracy: 27.49% → 27.07%
- Median group accuracy: 15.38% → 11.11%
- Zero-accuracy groups: 24/55 → 25/55

At group level, 10 groups improved, 13 degraded, and 32 remained
unchanged. Only one previously zero-accuracy group was recovered,
while two additional groups collapsed to zero accuracy.

The intervention therefore redistributed performance across unseen
groups rather than producing a systematic improvement in
generalization. Increasing fine-tuning depth alone is consequently
not the next preferred intervention.

## Mitigation experiment 2 — Group-balanced training

The baseline gives each training image equal importance, meaning that
large split groups contribute more strongly to the optimization objective
than small groups.

This experiment tests whether balancing the contribution of training
groups improves generalization to unseen groups.

In [45]:
# ============================================================
# Mitigation experiment 2
# Step 1 — Training group-size distribution
# ============================================================

train_group_sizes = (
    train_df
    .groupby(["pathology", "split_group"])
    .size()
    .rename("n_images")
    .reset_index()
)

print("Training groups:", len(train_group_sizes))
print()
print(train_group_sizes["n_images"].describe())

print("\nSmallest groups:")
display(
    train_group_sizes
    .sort_values("n_images")
    .head(10)
)

print("\nLargest groups:")
display(
    train_group_sizes
    .sort_values("n_images", ascending=False)
    .head(10)
)

Training groups: 247

count    247.000000
mean      32.097166
std       42.512058
min        1.000000
25%       11.000000
50%       17.000000
75%       32.000000
max      310.000000
Name: n_images, dtype: float64

Smallest groups:


,pathology,split_group,n_images
34,Astrocytoma,52,1
241,Schwannoma,351,1
14,Astrocytoma,32,2
177,Oligodendroglioma,257,3
189,Other,287,3
171,Oligodendroglioma,251,3
118,Meningioma,182,4
183,Other,280,4
182,Other,279,4
209,Other,307,4



Largest groups:


,pathology,split_group,n_images
154,Normal,222,310
43,Ependymoma,64,276
225,Schwannoma,335,226
96,Hemangiopericytoma,140,192
56,Glioma,83,185
155,Normal,226,163
180,Other,275,149
124,Meningioma,188,138
181,Other,276,138
226,Schwannoma,336,136


In [46]:
# ============================================================
# Mitigation experiment 2
# Step 2 — Compute moderate group-balanced sample weights
# ============================================================

# Number of training images in each split_group
group_size_map = (
    train_df
    .groupby("split_group")
    .size()
    .to_dict()
)

train_weighted_df = train_df.copy()

train_weighted_df["group_size"] = (
    train_weighted_df["split_group"]
    .map(group_size_map)
)

# Moderate inverse-frequency weighting
train_weighted_df["sample_weight"] = (
    1.0 / np.sqrt(train_weighted_df["group_size"])
)

# Normalize so that mean sample weight = 1
train_weighted_df["sample_weight"] /= (
    train_weighted_df["sample_weight"].mean()
)

print(
    train_weighted_df[
        ["group_size", "sample_weight"]
    ].describe()
)

print("\nSmallest training groups:")
display(
    train_weighted_df[
        ["pathology", "split_group", "group_size", "sample_weight"]
    ]
    .drop_duplicates()
    .sort_values("group_size")
    .head(10)
)

print("\nLargest training groups:")
display(
    train_weighted_df[
        ["pathology", "split_group", "group_size", "sample_weight"]
    ]
    .drop_duplicates()
    .sort_values("group_size", ascending=False)
    .head(10)
)

       group_size  sample_weight
count  7928.00000    7928.000000
mean     88.17558       1.000000
std      81.31511       0.549595
min       1.00000       0.367165
25%      22.00000       0.560552
50%      64.00000       0.808075
75%     133.00000       1.378257
max     310.00000       6.464599

Smallest training groups:


,pathology,split_group,group_size,sample_weight
2412,Astrocytoma,52,1,6.464599
9539,Schwannoma,351,1,6.464599
1941,Astrocytoma,32,2,4.571162
7636,Oligodendroglioma,257,3,3.732338
8025,Other,287,3,3.732338
7566,Oligodendroglioma,251,3,3.732338
5410,Meningioma,182,4,3.232299
7953,Other,280,4,3.232299
7949,Other,279,4,3.232299
8476,Other,307,4,3.232299



Largest training groups:


,pathology,split_group,group_size,sample_weight
6527,Normal,222,310,0.367165
2536,Ependymoma,64,276,0.389123
8895,Schwannoma,335,226,0.430019
4437,Hemangiopericytoma,140,192,0.466542
3180,Glioma,83,185,0.475287
6837,Normal,226,163,0.506346
7662,Other,275,149,0.529601
5620,Meningioma,188,138,0.550303
7811,Other,276,138,0.550303
9121,Schwannoma,336,136,0.554335


In [47]:
# ============================================================
# Mitigation experiment 2
# Step 3 — Weighted training dataset
# ============================================================

def build_weighted_train_dataset(df):
    paths = make_absolute_paths(df)

    labels = np.array(
        [class_to_idx[x] for x in df["pathology"]],
        dtype=np.int32
    )

    weights = df["sample_weight"].to_numpy(dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels, weights)
    )

    ds = ds.shuffle(
        buffer_size=len(df),
        seed=42,
        reshuffle_each_iteration=True
    )

    def load_weighted_image(path, label, weight):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32)

        return image, label, weight

    ds = ds.map(
        load_weighted_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


weighted_train_ds = build_weighted_train_dataset(
    train_weighted_df
)

print("Weighted train batches:", len(weighted_train_ds))

images, labels, weights = next(iter(weighted_train_ds))

print("Images :", images.shape)
print("Labels :", labels.shape)
print("Weights:", weights.shape)

print("\nWeight sanity check:")
print("min :", float(tf.reduce_min(weights)))
print("mean:", float(tf.reduce_mean(weights)))
print("max :", float(tf.reduce_max(weights)))

print("\nFirst 10 weights:")
print(weights[:10].numpy())

Weighted train batches: 248
Images : (32, 224, 224, 3)
Labels : (32,)
Weights: (32,)

Weight sanity check:
min : 0.3671645522117615
mean: 1.1138525009155273
max : 2.891056537628174

First 10 weights:
[0.55433476 0.36716455 0.6108472  0.77266735 0.7095819  1.3195808
 2.8910565  2.1548662  0.6108472  0.43001872]


In [48]:
# ============================================================
# Mitigation experiment 2
# Step 4 — Load fresh baseline
# ============================================================

MITIGATION_2_MODEL_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

mitigation_model_2 = tf.keras.models.load_model(
    MITIGATION_2_MODEL_PATH
)

print("✓ Fresh baseline loaded for Mitigation 2")
print("Parameters:", f"{mitigation_model_2.count_params():,}")

✓ Fresh baseline loaded for Mitigation 2
Parameters: 5,932,122


In [49]:
# ============================================================
# Step 5 — Conservative fine-tuning
# Same configuration as Mitigation 1
# ============================================================

backbone_2 = mitigation_model_2.get_layer("efficientnetv2-b0")

# Freeze everything first
backbone_2.trainable = True

for layer in backbone_2.layers:
    layer.trainable = False

# Unfreeze block6h + top_conv, except BatchNormalization
for layer in backbone_2.layers:
    if layer.name.startswith("block6h") or layer.name == "top_conv":
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True


trainable_backbone_layers = [
    layer.name
    for layer in backbone_2.layers
    if layer.trainable
]

trainable_bn = [
    layer.name
    for layer in backbone_2.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
    and layer.trainable
]

print("Trainable backbone layers:")
for name in trainable_backbone_layers:
    print(" -", name)

print("\nNumber of trainable backbone layers:",
      len(trainable_backbone_layers))

print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in mitigation_model_2.trainable_weights):,}"
)

print("Trainable BatchNorm layers:", trainable_bn)

Trainable backbone layers:
 - block6h_expand_conv
 - block6h_expand_activation
 - block6h_dwconv2
 - block6h_activation
 - block6h_se_squeeze
 - block6h_se_reshape
 - block6h_se_reduce
 - block6h_se_expand
 - block6h_se_excite
 - block6h_project_conv
 - block6h_drop
 - block6h_add
 - top_conv

Number of trainable backbone layers: 13
Trainable parameters: 823,098
Trainable BatchNorm layers: []


In [50]:
# ============================================================
# Step 6 — Compile Mitigation 2
# ============================================================

mitigation_model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("✓ Mitigation 2 compiled")
print(
    "Learning rate:",
    float(tf.keras.backend.get_value(
        mitigation_model_2.optimizer.learning_rate
    ))
)
print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in mitigation_model_2.trainable_weights):,}"
)

✓ Mitigation 2 compiled
Learning rate: 9.999999747378752e-06
Trainable parameters: 823,098


In [51]:
# ============================================================
# Step 7 — Callbacks
# ============================================================

MITIGATION_2_BEST_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_mitigation2_best.keras"
)

callbacks_2 = [
    tf.keras.callbacks.ModelCheckpoint(
        MITIGATION_2_BEST_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Mitigation 2 callbacks ready")
print("Best model will be saved to:", MITIGATION_2_BEST_PATH)

✓ Mitigation 2 callbacks ready
Best model will be saved to: /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation2_best.keras


In [52]:
# ============================================================
# Step 8 — Train Mitigation 2
# Group-aware sample weighting
# ============================================================

history_mitigation_2 = mitigation_model_2.fit(
    weighted_train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_2,
    verbose=1
)

Epoch 1/15
247/248 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.5013 - loss: 1.6292
Epoch 1: val_loss improved from None to 2.04196, saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation2_best.keras
248/248 ━━━━━━━━━━━━━━━━━━━━ 40s 110ms/step - accuracy: 0.4946 - loss: 1.6199 - val_accuracy: 0.2849 - val_loss: 2.0420 - learning_rate: 1.0000e-05
Epoch 2/15
247/248 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.5197 - loss: 1.5967
Epoch 2: val_loss improved from 2.04196 to 2.03572, saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation2_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation2_best.keras
248/248 ━━━━━━━━━━━━━━━━━━━━ 24s 98ms/step - accuracy: 0.5143 - loss: 1.5690 - val_accuracy: 0.2837 - val_loss: 2.0357 - learning_r

In [53]:
# ============================================================
# Step 9 — Evaluate best Mitigation 2 checkpoint
# ============================================================

best_mitigation_model_2 = tf.keras.models.load_model(
    MITIGATION_2_BEST_PATH
)

print("✓ Best Mitigation 2 checkpoint loaded")

val_loss_2, val_acc_2 = best_mitigation_model_2.evaluate(
    val_ds,
    verbose=1
)

test_loss_2, test_acc_2 = best_mitigation_model_2.evaluate(
    test_ds,
    verbose=1
)

print("\nMitigation experiment 2")
print(f"Validation accuracy : {val_acc_2:.4f} ({100*val_acc_2:.2f}%)")
print(f"Validation loss     : {val_loss_2:.4f}")
print(f"Test accuracy       : {test_acc_2:.4f} ({100*test_acc_2:.2f}%)")
print(f"Test loss           : {test_loss_2:.4f}")

✓ Best Mitigation 2 checkpoint loaded
54/54 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.2843 - loss: 2.0236
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.3096 - loss: 2.0158

Mitigation experiment 2
Validation accuracy : 0.2843 (28.43%)
Validation loss     : 2.0236
Test accuracy       : 0.3096 (30.96%)
Test loss           : 2.0158


In [54]:
# ============================================================
# Step 10 — Group-level evaluation: Mitigation 2
# ============================================================

# Validation predictions
mitigation_2_probs = best_mitigation_model_2.predict(
    val_ds,
    verbose=1
)

mitigation_2_pred = np.argmax(
    mitigation_2_probs,
    axis=1
)

# Sanity check
print("\nPredictions:", len(mitigation_2_pred))
print("Labels     :", len(y_true_val))
print(
    "Accuracy   :",
    round((mitigation_2_pred == y_true_val).mean(), 4)
)

# Attach predictions to validation metadata
val_mitigation_2 = val_df.copy()

val_mitigation_2["y_true"] = y_true_val
val_mitigation_2["y_pred"] = mitigation_2_pred
val_mitigation_2["correct"] = (
    val_mitigation_2["y_true"]
    == val_mitigation_2["y_pred"]
).astype(int)

# Group-level results
mitigation_2_group_results = (
    val_mitigation_2
    .groupby(["pathology", "split_group"])
    .agg(
        n_images=("correct", "size"),
        n_correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

# Summary
mitigation_2_group_summary = {
    "image_accuracy":
        val_mitigation_2["correct"].mean(),

    "macro_group_accuracy":
        mitigation_2_group_results["accuracy"].mean(),

    "median_group_accuracy":
        mitigation_2_group_results["accuracy"].median(),

    "zero_accuracy_groups":
        int(
            (mitigation_2_group_results["accuracy"] == 0).sum()
        ),

    "n_groups":
        len(mitigation_2_group_results)
}

pd.Series(mitigation_2_group_summary)

54/54 ━━━━━━━━━━━━━━━━━━━━ 13s 172ms/step

Predictions: 1699
Labels     : 1699
Accuracy   : 0.2843


,0
image_accuracy,0.284285
macro_group_accuracy,0.267850
median_group_accuracy,0.111111
zero_accuracy_groups,23.000000
n_groups,55.000000


In [56]:
for var in [
    "baseline_group_results",
    "glioma_groups",
    "group_results",
    "mitigation_group_results",
    "mitigation_2_group_results"
]:
    print(f"{var:30s} ->", var in globals())

baseline_group_results         -> False
glioma_groups                  -> False
group_results                  -> False
mitigation_group_results       -> True
mitigation_2_group_results     -> True


In [57]:
# ============================================================
# Step 11A — Rebuild baseline group results
# ============================================================

baseline_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

baseline_probs = baseline_model.predict(
    val_ds,
    verbose=1
)

baseline_pred = np.argmax(
    baseline_probs,
    axis=1
)

print("\nPredictions:", len(baseline_pred))
print("Labels     :", len(y_true_val))
print(
    "Accuracy   :",
    round((baseline_pred == y_true_val).mean(), 4)
)

val_baseline = val_df.copy()

val_baseline["y_true"] = y_true_val
val_baseline["y_pred"] = baseline_pred
val_baseline["correct"] = (
    val_baseline["y_true"] == val_baseline["y_pred"]
).astype(int)

baseline_group_results = (
    val_baseline
    .groupby(["pathology", "split_group"])
    .agg(
        n_images=("correct", "size"),
        n_correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

baseline_group_summary = {
    "image_accuracy": val_baseline["correct"].mean(),
    "macro_group_accuracy":
        baseline_group_results["accuracy"].mean(),
    "median_group_accuracy":
        baseline_group_results["accuracy"].median(),
    "zero_accuracy_groups":
        int((baseline_group_results["accuracy"] == 0).sum()),
    "n_groups":
        len(baseline_group_results)
}

pd.Series(baseline_group_summary)

54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 110ms/step

Predictions: 1699
Labels     : 1699
Accuracy   : 0.3014


,0
image_accuracy,0.301354
macro_group_accuracy,0.274869
median_group_accuracy,0.153846
zero_accuracy_groups,24.000000
n_groups,55.000000


In [58]:
# ============================================================
# Step 11 — Baseline vs Mitigation 1 vs Mitigation 2
# Group-level comparison
# ============================================================

comparison_3 = (
    baseline_group_results[
        ["pathology", "split_group", "n_images", "accuracy"]
    ]
    .rename(columns={"accuracy": "baseline_accuracy"})
    .merge(
        mitigation_group_results[
            ["pathology", "split_group", "accuracy"]
        ].rename(columns={"accuracy": "mitigation1_accuracy"}),
        on=["pathology", "split_group"],
        how="inner"
    )
    .merge(
        mitigation_2_group_results[
            ["pathology", "split_group", "accuracy"]
        ].rename(columns={"accuracy": "mitigation2_accuracy"}),
        on=["pathology", "split_group"],
        how="inner"
    )
)

for col in [
    "baseline_accuracy",
    "mitigation1_accuracy",
    "mitigation2_accuracy"
]:
    comparison_3[col + "_pct"] = 100 * comparison_3[col]

comparison_3["delta_m2_vs_baseline"] = 100 * (
    comparison_3["mitigation2_accuracy"]
    - comparison_3["baseline_accuracy"]
)

comparison_3["delta_m2_vs_m1"] = 100 * (
    comparison_3["mitigation2_accuracy"]
    - comparison_3["mitigation1_accuracy"]
)

# Groups rescued relative to baseline
rescued_m2 = comparison_3[
    (comparison_3["baseline_accuracy"] == 0) &
    (comparison_3["mitigation2_accuracy"] > 0)
].copy()

# Groups that were >0 in baseline but collapse to 0
collapsed_m2 = comparison_3[
    (comparison_3["baseline_accuracy"] > 0) &
    (comparison_3["mitigation2_accuracy"] == 0)
].copy()

print("Groups rescued vs baseline:", len(rescued_m2))
display(
    rescued_m2[
        [
            "pathology", "split_group", "n_images",
            "baseline_accuracy_pct",
            "mitigation1_accuracy_pct",
            "mitigation2_accuracy_pct"
        ]
    ]
)

print("\nGroups newly collapsed vs baseline:", len(collapsed_m2))
display(
    collapsed_m2[
        [
            "pathology", "split_group", "n_images",
            "baseline_accuracy_pct",
            "mitigation1_accuracy_pct",
            "mitigation2_accuracy_pct"
        ]
    ]
)

print("\nLargest improvements vs baseline:")
display(
    comparison_3.sort_values(
        "delta_m2_vs_baseline",
        ascending=False
    )[
        [
            "pathology", "split_group", "n_images",
            "baseline_accuracy_pct",
            "mitigation2_accuracy_pct",
            "delta_m2_vs_baseline"
        ]
    ].head(10)
)

print("\nLargest degradations vs baseline:")
display(
    comparison_3.sort_values(
        "delta_m2_vs_baseline"
    )[
        [
            "pathology", "split_group", "n_images",
            "baseline_accuracy_pct",
            "mitigation2_accuracy_pct",
            "delta_m2_vs_baseline"
        ]
    ].head(10)
)

Groups rescued vs baseline: 2


,pathology,split_group,n_images,baseline_accuracy_pct,mitigation1_accuracy_pct,mitigation2_accuracy_pct
6,Astrocytoma,7,29,0.0,0.000000,3.448276
37,Oligodendroglioma,240,23,0.0,17.391304,17.391304



Groups newly collapsed vs baseline: 1


,pathology,split_group,n_images,baseline_accuracy_pct,mitigation1_accuracy_pct,mitigation2_accuracy_pct
31,Meningioma,156,99,16.161616,0.0,0.0



Largest improvements vs baseline:


,pathology,split_group,n_images,baseline_accuracy_pct,mitigation2_accuracy_pct,delta_m2_vs_baseline
38,Oligodendroglioma,241,26,15.384615,50.000000,34.615385
47,Other,270,34,35.294118,55.882353,20.588235
37,Oligodendroglioma,240,23,0.000000,17.391304,17.391304
35,Normal,227,55,16.363636,25.454545,9.090909
53,Schwannoma,326,36,50.000000,58.333333,8.333333
48,Other,272,73,2.739726,8.219178,5.479452
24,Meningioma,146,27,74.074074,77.777778,3.703704
6,Astrocytoma,7,29,0.000000,3.448276,3.448276
15,Glioma,86,29,17.241379,20.689655,3.448276
34,Normal,224,105,59.047619,61.904762,2.857143



Largest degradations vs baseline:


,pathology,split_group,n_images,baseline_accuracy_pct,mitigation2_accuracy_pct,delta_m2_vs_baseline
10,Ependymoma,60,50,24.000000,2.000000,-22.000000
31,Meningioma,156,99,16.161616,0.000000,-16.161616
28,Meningioma,151,13,76.923077,61.538462,-15.384615
9,Ependymoma,57,13,38.461538,23.076923,-15.384615
11,Ependymoma,61,93,60.215054,46.236559,-13.978495
29,Meningioma,153,37,45.945946,32.432432,-13.513514
13,Glioma,82,30,36.666667,23.333333,-13.333333
30,Meningioma,154,30,90.000000,80.000000,-10.000000
16,Glioma,87,33,48.484848,39.393939,-9.090909
49,Schwannoma,318,42,83.333333,76.190476,-7.142857


## Pixel-shift analysis before Mitigation 3

Before introducing a third mitigation strategy, we investigate whether
poor generalization across unseen groups is associated with systematic
differences in image intensity distributions.

In [59]:
print(manifest.columns.tolist())

['relative_path', 'resolved_relative_path', 'pathology', 'modality', 'split_group', 'split', 'width', 'height', 'point', 'location', 'description', 'pixel_hash']


In [60]:
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def compute_pixel_stats(df):
    rows = []

    for idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Computing pixel statistics"
    ):
        img_path = PROJECT_ROOT / row["resolved_relative_path"]

        # MRI intensity analysis in grayscale
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        pixels = img.astype(np.float32).ravel()

        rows.append({
            "index": idx,
            "pathology": row["pathology"],
            "split_group": row["split_group"],
            "split": row["split"],

            "pixel_mean": pixels.mean(),
            "pixel_std": pixels.std(),

            "pixel_p05": np.percentile(pixels, 5),
            "pixel_p25": np.percentile(pixels, 25),
            "pixel_median": np.percentile(pixels, 50),
            "pixel_p75": np.percentile(pixels, 75),
            "pixel_p95": np.percentile(pixels, 95),
        })

    return pd.DataFrame(rows)


pixel_stats = compute_pixel_stats(manifest)

print("Images analysed:", len(pixel_stats))
print("\nSplit counts:")
print(pixel_stats["split"].value_counts())

pixel_stats.head()

Computing pixel statistics:   0%|          | 0/11300 [00:00<?, ?it/s]

Images analysed: 11300

Split counts:
split
train         7928
validation    1699
test          1673
Name: count, dtype: int64


,index,pathology,split_group,split,pixel_mean,pixel_std,pixel_p05,pixel_p25,pixel_median,pixel_p75,pixel_p95
0,0,Astrocytoma,9,test,59.721760,64.067215,1.0,2.0,17.0,113.0,181.0
1,1,Astrocytoma,9,test,58.950066,63.754734,1.0,2.0,12.0,112.0,181.0
2,2,Astrocytoma,9,test,58.107075,63.057121,1.0,2.0,9.0,110.0,179.0
3,3,Astrocytoma,9,test,57.815197,62.980694,2.0,2.0,7.0,110.0,179.0
4,4,Astrocytoma,9,test,57.714924,50.647930,23.0,24.0,28.0,81.0,167.0


In [61]:
# ============================================================
# Validation group pixel profiles + baseline accuracy
# ============================================================

PIXEL_FEATURES = [
    "pixel_mean",
    "pixel_std",
    "pixel_p05",
    "pixel_p25",
    "pixel_median",
    "pixel_p75",
    "pixel_p95",
]

# ------------------------------------------------------------
# 1. Validation pixel statistics aggregated by split_group
# ------------------------------------------------------------

val_pixel_groups = (
    pixel_stats[pixel_stats["split"] == "validation"]
    .groupby(["pathology", "split_group"])
    .agg(
        n_images_pixel=("index", "size"),
        **{
            f"median_{feature}": (feature, "median")
            for feature in PIXEL_FEATURES
        }
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Attach baseline group performance
# ------------------------------------------------------------

val_pixel_vs_accuracy = (
    baseline_group_results[
        ["pathology", "split_group", "n_images", "accuracy"]
    ]
    .merge(
        val_pixel_groups,
        on=["pathology", "split_group"],
        how="left"
    )
)

val_pixel_vs_accuracy["accuracy_pct"] = (
    100 * val_pixel_vs_accuracy["accuracy"]
)

print("Validation groups:", len(val_pixel_vs_accuracy))
print(
    "Groups with missing pixel statistics:",
    val_pixel_vs_accuracy[
        [f"median_{f}" for f in PIXEL_FEATURES]
    ].isna().any(axis=1).sum()
)

display(
    val_pixel_vs_accuracy.sort_values("accuracy")[
        [
            "pathology",
            "split_group",
            "n_images",
            "accuracy_pct",
            "median_pixel_mean",
            "median_pixel_std",
            "median_pixel_p05",
            "median_pixel_median",
            "median_pixel_p95",
        ]
    ]
)

Validation groups: 55
Groups with missing pixel statistics: 0


,pathology,split_group,n_images,accuracy_pct,median_pixel_mean,median_pixel_std,median_pixel_p05,median_pixel_median,median_pixel_p95
0,Astrocytoma,0,11,0.000000,43.866726,55.827377,0.0,7.0,138.0
1,Astrocytoma,2,6,0.000000,74.623070,70.811157,1.0,77.5,211.0
2,Astrocytoma,3,53,0.000000,39.848370,50.078781,8.0,9.0,123.0
3,Astrocytoma,4,47,0.000000,44.719303,51.696129,6.0,8.0,125.0
4,Astrocytoma,5,3,0.000000,57.380543,70.795998,0.0,6.0,206.0
5,Astrocytoma,6,4,0.000000,66.934784,62.259140,0.0,76.0,151.0
6,Astrocytoma,7,29,0.000000,43.309299,50.768383,0.0,9.0,115.0
7,Astrocytoma,8,3,0.000000,36.327953,44.801472,1.0,5.0,98.0
8,Astrocytoma,10,12,0.000000,34.584801,40.122265,0.0,5.0,86.0
12,Glioma,81,47,0.000000,48.818798,54.493164,0.0,15.0,135.0


In [62]:
# ============================================================
# Pixel shift relative to training groups
# ============================================================

from scipy.stats import spearmanr

PIXEL_FEATURES = [
    "pixel_mean",
    "pixel_std",
    "pixel_p05",
    "pixel_p25",
    "pixel_median",
    "pixel_p75",
    "pixel_p95",
]

GROUP_FEATURES = [f"median_{f}" for f in PIXEL_FEATURES]


# ------------------------------------------------------------
# 1. Build pixel profiles for TRAIN groups
# ------------------------------------------------------------

train_pixel_groups = (
    pixel_stats[pixel_stats["split"] == "train"]
    .groupby(["pathology", "split_group"])
    .agg(
        n_images_pixel=("index", "size"),
        **{
            f"median_{feature}": (feature, "median")
            for feature in PIXEL_FEATURES
        }
    )
    .reset_index()
)

print("Train groups:", len(train_pixel_groups))


# ------------------------------------------------------------
# 2. Training reference distribution for each pathology
# ------------------------------------------------------------

train_reference = (
    train_pixel_groups
    .groupby("pathology")[GROUP_FEATURES]
    .agg(["median", "std"])
)

rows = []

for _, row in val_pixel_vs_accuracy.iterrows():

    pathology = row["pathology"]

    # Typical training profile for this pathology
    center = train_reference.loc[pathology].xs(
        "median",
        level=1
    )

    scale = train_reference.loc[pathology].xs(
        "std",
        level=1
    )

    # Avoid division by zero
    scale = scale.replace(0, np.nan)

    z = (
        row[GROUP_FEATURES].astype(float) - center
    ) / scale

    # Mean absolute standardized distance
    pixel_shift_distance = np.nanmean(np.abs(z.values))

    rows.append(pixel_shift_distance)


val_pixel_vs_accuracy["pixel_shift_distance"] = rows


# ------------------------------------------------------------
# 3. Association with baseline accuracy
# ------------------------------------------------------------

rho, p_value = spearmanr(
    val_pixel_vs_accuracy["pixel_shift_distance"],
    val_pixel_vs_accuracy["accuracy"]
)

print("\nSpearman correlation")
print("--------------------")
print(f"rho     : {rho:.3f}")
print(f"p-value : {p_value:.4f}")


# ------------------------------------------------------------
# 4. Inspect groups with largest pixel shift
# ------------------------------------------------------------

display(
    val_pixel_vs_accuracy.sort_values(
        "pixel_shift_distance",
        ascending=False
    )[
        [
            "pathology",
            "split_group",
            "n_images",
            "accuracy_pct",
            "pixel_shift_distance",
            "median_pixel_mean",
            "median_pixel_std",
            "median_pixel_median",
            "median_pixel_p95",
        ]
    ].head(20)
)

Train groups: 247

Spearman correlation
--------------------
rho     : 0.237
p-value : 0.0818


,pathology,split_group,n_images,accuracy_pct,pixel_shift_distance,median_pixel_mean,median_pixel_std,median_pixel_median,median_pixel_p95
1,Astrocytoma,2,6,0.000000,2.519142,74.623070,70.811157,77.5,211.0
5,Astrocytoma,6,4,0.000000,1.942227,66.934784,62.259140,76.0,151.0
20,Hemangiopericytoma,133,16,0.000000,1.654132,59.807777,46.444405,22.0,147.5
4,Astrocytoma,5,3,0.000000,1.538038,57.380543,70.795998,6.0,206.0
51,Schwannoma,322,25,68.000000,1.515849,59.076481,62.933044,24.0,189.0
29,Meningioma,153,37,45.945946,1.470122,48.863232,51.543835,16.0,129.0
36,Oligodendroglioma,239,19,0.000000,1.401919,18.099056,28.096094,2.0,77.0
28,Meningioma,151,13,76.923077,1.218263,54.687511,70.416000,15.0,190.0
54,Schwannoma,327,16,62.500000,1.190999,38.072941,41.814911,16.5,119.5
39,Oligodendroglioma,242,14,0.000000,1.111411,46.766914,49.678734,15.0,145.0


In [63]:
# ============================================================
# Feature-wise pixel association with accuracy
# ============================================================

from scipy.stats import spearmanr
import pandas as pd

results = []

for feature in GROUP_FEATURES:

    rho, p_value = spearmanr(
        val_pixel_vs_accuracy[feature],
        val_pixel_vs_accuracy["accuracy"]
    )

    results.append({
        "feature": feature,
        "spearman_rho": rho,
        "p_value": p_value
    })

pixel_feature_correlations = (
    pd.DataFrame(results)
    .sort_values("p_value")
    .reset_index(drop=True)
)

display(pixel_feature_correlations)

,feature,spearman_rho,p_value
0,median_pixel_p75,-0.242268,0.074735
1,median_pixel_std,-0.215951,0.113307
2,median_pixel_mean,-0.166635,0.224009
3,median_pixel_p25,-0.127842,0.352293
4,median_pixel_p95,-0.104681,0.446898
5,median_pixel_median,0.093769,0.495911
6,median_pixel_p05,-0.016418,0.905299


In [64]:
# ============================================================
# Feature-wise SHIFT vs accuracy
# ============================================================

shift_results = []

for feature in GROUP_FEATURES:

    shifts = []

    for _, row in val_pixel_vs_accuracy.iterrows():

        pathology = row["pathology"]

        center = train_reference.loc[pathology].loc[
            feature, "median"
        ]

        scale = train_reference.loc[pathology].loc[
            feature, "std"
        ]

        if pd.isna(scale) or scale == 0:
            shifts.append(np.nan)
        else:
            shifts.append(
                abs((row[feature] - center) / scale)
            )

    temp = pd.DataFrame({
        "shift": shifts,
        "accuracy": val_pixel_vs_accuracy["accuracy"]
    }).dropna()

    rho, p_value = spearmanr(
        temp["shift"],
        temp["accuracy"]
    )

    shift_results.append({
        "feature": feature,
        "spearman_rho": rho,
        "p_value": p_value
    })

pixel_shift_correlations = (
    pd.DataFrame(shift_results)
    .sort_values("p_value")
    .reset_index(drop=True)
)

display(pixel_shift_correlations)

,feature,spearman_rho,p_value
0,median_pixel_p95,0.234753,0.084493
1,median_pixel_p75,0.177144,0.195729
2,median_pixel_std,0.154918,0.258750
3,median_pixel_mean,0.125494,0.361279
4,median_pixel_median,0.056525,0.681878
5,median_pixel_p25,-0.013297,0.923240
6,median_pixel_p05,0.004878,0.971802


### Pixel-shift diagnosis

Global intensity statistics did not provide evidence that validation-group
failure was driven by systematic pixel-distribution shift.

The aggregate distance between each validation group and the corresponding
training pathology profile showed only a weak positive association with
group accuracy (Spearman ρ = 0.237, p = 0.082), contrary to the expected
negative relationship.

Feature-wise analyses led to the same conclusion. Raw upper-intensity
statistics showed a weak negative trend for the 75th percentile
(ρ = -0.242, p = 0.075), but no individual intensity feature showed a
statistically significant association with group accuracy.

Moreover, feature-specific distances from the training distribution were
not negatively associated with performance.

These results suggest that simple global intensity shift is unlikely to be
the primary driver of the observed cross-group generalization failures.
Consequently, histogram equalization is not prioritized as the next
mitigation strategy.

## Mitigation experiment 3 — Group-aware sampling

In [65]:
# ============================================================
# Step 1 — Training group structure by pathology
# ============================================================

train_group_structure = (
    train_df
    .groupby(["pathology", "split_group"])
    .size()
    .reset_index(name="n_images")
)

group_summary_by_pathology = (
    train_group_structure
    .groupby("pathology")
    .agg(
        n_groups=("split_group", "nunique"),
        n_images=("n_images", "sum"),
        min_group_size=("n_images", "min"),
        median_group_size=("n_images", "median"),
        max_group_size=("n_images", "max"),
    )
    .sort_index()
)

display(group_summary_by_pathology)

print(
    "\nTotal:",
    train_group_structure["split_group"].nunique(),
    "training groups"
)

,n_groups,n_images,min_group_size,median_group_size,max_group_size
pathology,,,,,
Astrocytoma,39,782,1,16.0,80
Ependymoma,17,725,11,23.0,276
Glioma,35,1034,5,17.0,185
Hemangiopericytoma,7,428,5,15.0,192
Meningioma,45,1449,4,17.0,138
Neurocytoma,11,436,12,24.0,120
Normal,10,750,15,32.5,310
Oligodendroglioma,16,385,3,14.5,100
Other,39,1073,3,15.0,149



Total: 247 training groups


In [66]:
# ============================================================
# Step 2 — Moderate group-aware sampling probabilities
# ============================================================

train_sampling_df = train_df.copy()

# Size of each split_group
group_sizes = (
    train_sampling_df
    .groupby("split_group")
    .size()
)

train_sampling_df["group_size"] = (
    train_sampling_df["split_group"]
    .map(group_sizes)
)

# Moderate inverse-sqrt correction
train_sampling_df["sampling_weight"] = (
    1.0 / np.sqrt(train_sampling_df["group_size"])
)

# Normalize into probabilities
train_sampling_df["sampling_probability"] = (
    train_sampling_df["sampling_weight"]
    / train_sampling_df["sampling_weight"].sum()
)

print("Images:", len(train_sampling_df))
print("Groups:", train_sampling_df["split_group"].nunique())

print("\nSampling weight:")
print(train_sampling_df["sampling_weight"].describe())

print("\nProbability sum:",
      train_sampling_df["sampling_probability"].sum())

Images: 7928
Groups: 247

Sampling weight:
count    7928.000000
mean        0.154689
std         0.085016
min         0.056796
25%         0.086711
50%         0.125000
75%         0.213201
max         1.000000
Name: sampling_weight, dtype: float64

Probability sum: 1.0


In [67]:
# Effective probability mass assigned to each group

group_sampling_mass = (
    train_sampling_df
    .groupby(["pathology", "split_group"])
    .agg(
        n_images=("relative_path", "size"),
        sampling_mass=("sampling_probability", "sum")
    )
    .reset_index()
)

print("\nGroup sampling mass:")
display(
    group_sampling_mass
    .sort_values("sampling_mass")
    .head(10)
)

display(
    group_sampling_mass
    .sort_values("sampling_mass", ascending=False)
    .head(10)
)

print("\nSampling mass ratio:")
print(
    group_sampling_mass["sampling_mass"].max()
    / group_sampling_mass["sampling_mass"].min()
)


Group sampling mass:


,pathology,split_group,n_images,sampling_mass
34,Astrocytoma,52,1,0.000815
241,Schwannoma,351,1,0.000815
14,Astrocytoma,32,2,0.001153
177,Oligodendroglioma,257,3,0.001412
189,Other,287,3,0.001412
171,Oligodendroglioma,251,3,0.001412
118,Meningioma,182,4,0.001631
183,Other,280,4,0.001631
182,Other,279,4,0.001631
209,Other,307,4,0.001631


,pathology,split_group,n_images,sampling_mass
154,Normal,222,310,0.014357
43,Ependymoma,64,276,0.013547
225,Schwannoma,335,226,0.012258
96,Hemangiopericytoma,140,192,0.011299
56,Glioma,83,185,0.011091
155,Normal,226,163,0.010411
180,Other,275,149,0.009953
124,Meningioma,188,138,0.009579
181,Other,276,138,0.009579
226,Schwannoma,336,136,0.009509



Sampling mass ratio:
17.60681686165901


In [70]:
# ============================================================
# Step 3 — Build group-aware training dataset
# ============================================================

N_TRAIN = len(train_sampling_df)
SEED = 42

# Paths + labels
sampling_paths = np.array(
    make_absolute_paths(train_sampling_df),
    dtype=str
)

sampling_labels = np.array(
    [class_to_idx[x] for x in train_sampling_df["pathology"]],
    dtype=np.int32
)

sampling_probs = (
    train_sampling_df["sampling_probability"]
    .to_numpy(dtype=np.float64)
)


def group_aware_generator():
    """
    Sample N_TRAIN images per epoch with replacement,
    using inverse-sqrt group-aware probabilities.
    """
    rng = np.random.default_rng(SEED)

    while True:
        sampled_indices = rng.choice(
            N_TRAIN,
            size=N_TRAIN,
            replace=True,
            p=sampling_probs
        )

        rng.shuffle(sampled_indices)

        for idx in sampled_indices:
            yield sampling_paths[idx], sampling_labels[idx]


group_aware_train_ds = tf.data.Dataset.from_generator(
    group_aware_generator,
    output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.string),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )
)

# EXACT SAME preprocessing as build_dataset()
group_aware_train_ds = group_aware_train_ds.map(
    load_mri,
    num_parallel_calls=AUTOTUNE
)

group_aware_train_ds = group_aware_train_ds.batch(BATCH_SIZE)
group_aware_train_ds = group_aware_train_ds.prefetch(AUTOTUNE)

STEPS_PER_EPOCH = int(np.ceil(N_TRAIN / BATCH_SIZE))

print("✓ Group-aware training dataset created")
print("Images sampled per epoch:", N_TRAIN)
print("Steps per epoch:", STEPS_PER_EPOCH)

# Sanity check
images, labels = next(iter(group_aware_train_ds))

print("\nBatch sanity check")
print("Images :", images.shape)
print("Labels :", labels.shape)
print("dtype  :", images.dtype)
print(
    "range  :",
    float(tf.reduce_min(images)),
    "→",
    float(tf.reduce_max(images))
)

✓ Group-aware training dataset created
Images sampled per epoch: 7928
Steps per epoch: 248

Batch sanity check
Images : (32, 224, 224, 3)
Labels : (32,)
dtype  : <dtype: 'float32'>
range  : 0.0 → 255.0


In [71]:
# ============================================================
# Step 4 — Verify effective sampling over one epoch
# ============================================================

rng_check = np.random.default_rng(42)

sampled_indices_check = rng_check.choice(
    N_TRAIN,
    size=N_TRAIN,
    replace=True,
    p=sampling_probs
)

sampled_epoch_df = train_sampling_df.iloc[
    sampled_indices_check
].copy()

# Exposure per group
original_group_exposure = (
    train_sampling_df
    .groupby(["pathology", "split_group"])
    .size()
    .reset_index(name="original_n")
)

sampled_group_exposure = (
    sampled_epoch_df
    .groupby(["pathology", "split_group"])
    .size()
    .reset_index(name="sampled_n")
)

exposure_check = original_group_exposure.merge(
    sampled_group_exposure,
    on=["pathology", "split_group"],
    how="left"
)

exposure_check["sampled_n"] = (
    exposure_check["sampled_n"]
    .fillna(0)
    .astype(int)
)

print("Images drawn:", len(sampled_epoch_df))
print(
    "Groups represented:",
    (exposure_check["sampled_n"] > 0).sum(),
    "/",
    len(exposure_check)
)

print("\nSmallest original groups:")
display(
    exposure_check
    .sort_values(["original_n", "sampled_n"])
    .head(15)
)

print("\nLargest original groups:")
display(
    exposure_check
    .sort_values("original_n", ascending=False)
    .head(15)
)

# Exposure by pathology
pathology_exposure = (
    sampled_epoch_df["pathology"]
    .value_counts()
    .rename("sampled_n")
    .to_frame()
)

pathology_exposure["sampled_pct"] = (
    100 * pathology_exposure["sampled_n"] / N_TRAIN
)

print("\nExposure by pathology:")
display(pathology_exposure.sort_index())

Images drawn: 7928
Groups represented: 247 / 247

Smallest original groups:


,pathology,split_group,original_n,sampled_n
241,Schwannoma,351,1,6
34,Astrocytoma,52,1,8
14,Astrocytoma,32,2,8
177,Oligodendroglioma,257,3,12
171,Oligodendroglioma,251,3,18
189,Other,287,3,18
183,Other,280,4,6
182,Other,279,4,9
208,Other,306,4,12
213,Other,311,4,13



Largest original groups:


,pathology,split_group,original_n,sampled_n
154,Normal,222,310,97
43,Ependymoma,64,276,127
225,Schwannoma,335,226,94
96,Hemangiopericytoma,140,192,88
56,Glioma,83,185,86
155,Normal,226,163,83
180,Other,275,149,87
124,Meningioma,188,138,88
181,Other,276,138,85
226,Schwannoma,336,136,77



Exposure by pathology:


,sampled_n,sampled_pct
pathology,,
Astrocytoma,1054,13.294652
Ependymoma,666,8.400605
Glioma,1102,13.900101
Hemangiopericytoma,282,3.557013
Meningioma,1513,19.084258
Neurocytoma,419,5.285066
Normal,477,6.016650
Oligodendroglioma,429,5.411201
Other,1149,14.492936


In [72]:
# ============================================================
# Step 5 — Fresh model for Mitigation 3
# Group-aware sampling + conservative fine-tuning
# ============================================================

MITIGATION_3_MODEL_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_baseline.keras"
)

mitigation_3_model = tf.keras.models.load_model(
    MITIGATION_3_MODEL_PATH
)

print("✓ Fresh baseline loaded")
print("Parameters:", f"{mitigation_3_model.count_params():,}")

✓ Fresh baseline loaded
Parameters: 5,932,122


In [75]:
# ============================================================
# Step 6 — Conservative fine-tuning
# SAME configuration as Mitigation 1 and 2
# ============================================================

# Reload a clean baseline because the previous attempt
# modified trainable states in memory.
mitigation_3_model = tf.keras.models.load_model(
    MITIGATION_3_MODEL_PATH
)

backbone_3 = mitigation_3_model.get_layer("efficientnetv2-b0")

# The nested backbone itself must be trainable so that selected
# internal layers can receive gradients.
backbone_3.trainable = True

# Freeze EVERY internal layer explicitly
for layer in backbone_3.layers:
    layer.trainable = False

# Selectively unfreeze block6h + top_conv,
# while keeping all BatchNorm layers frozen.
for layer in backbone_3.layers:
    if layer.name.startswith("block6h") or layer.name == "top_conv":
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True


trainable_backbone_layers_3 = [
    layer.name
    for layer in backbone_3.layers
    if layer.trainable
]

trainable_bn_3 = [
    layer.name
    for layer in backbone_3.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
    and layer.trainable
]

trainable_params_3 = sum(
    np.prod(v.shape)
    for v in mitigation_3_model.trainable_weights
)

print("Trainable backbone layers:")
for name in trainable_backbone_layers_3:
    print(" -", name)

print(
    "\nNumber of trainable backbone layers:",
    len(trainable_backbone_layers_3)
)

print(
    "Trainable parameters:",
    f"{trainable_params_3:,}"
)

print(
    "Trainable BatchNorm layers:",
    trainable_bn_3
)

Trainable backbone layers:
 - block6h_expand_conv
 - block6h_expand_activation
 - block6h_dwconv2
 - block6h_activation
 - block6h_se_squeeze
 - block6h_se_reshape
 - block6h_se_reduce
 - block6h_se_expand
 - block6h_se_excite
 - block6h_project_conv
 - block6h_drop
 - block6h_add
 - top_conv

Number of trainable backbone layers: 13
Trainable parameters: 823,098
Trainable BatchNorm layers: []


In [76]:
# ============================================================
# Step 7 — Compile Mitigation 3
# ============================================================

mitigation_3_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("✓ Mitigation 3 compiled")
print(
    "Learning rate:",
    float(mitigation_3_model.optimizer.learning_rate.numpy())
)
print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in mitigation_3_model.trainable_weights):,}"
)

✓ Mitigation 3 compiled
Learning rate: 9.999999747378752e-06
Trainable parameters: 823,098


In [77]:
# ============================================================
# Step 8 — Mitigation 3 callbacks
# ============================================================

MITIGATION_3_BEST_PATH = (
    "/content/drive/MyDrive/brain-tumor-mri/models/"
    "efficientnetv2b0_mitigation3_best.keras"
)

mitigation_3_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MITIGATION_3_BEST_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Mitigation 3 callbacks ready")
print("Best model will be saved to:", MITIGATION_3_BEST_PATH)

✓ Mitigation 3 callbacks ready
Best model will be saved to: /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation3_best.keras


In [78]:
# ============================================================
# Step 9 — Train Mitigation 3
# ============================================================

history_mitigation_3 = mitigation_3_model.fit(
    group_aware_train_ds,
    validation_data=val_ds,

    epochs=15,
    steps_per_epoch=STEPS_PER_EPOCH,

    callbacks=mitigation_3_callbacks,
    verbose=1
)

Epoch 1/15
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4315 - loss: 1.6172
Epoch 1: val_loss improved from None to 2.03505, saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation3_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/brain-tumor-mri/models/efficientnetv2b0_mitigation3_best.keras
248/248 ━━━━━━━━━━━━━━━━━━━━ 41s 111ms/step - accuracy: 0.4311 - loss: 1.6121 - val_accuracy: 0.2808 - val_loss: 2.0351 - learning_rate: 1.0000e-05
Epoch 2/15
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.4491 - loss: 1.5666
Epoch 2: val_loss did not improve from 2.03505

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
248/248 ━━━━━━━━━━━━━━━━━━━━ 24s 97ms/step - accuracy: 0.4567 - loss: 1.5428 - val_accuracy: 0.2796 - val_loss: 2.0455 - learning_rate: 1.0000e-05
Epoch 3/15
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.4627 - loss: 1.5431
Epoch 3: val_loss did not improve from 2.03505



In [79]:
# ============================================================
# Step 10 — Load best Mitigation 3 and evaluate
# ============================================================

best_mitigation_3 = tf.keras.models.load_model(
    MITIGATION_3_BEST_PATH
)

print("✓ Best Mitigation 3 checkpoint loaded")

val_loss_m3, val_acc_m3 = best_mitigation_3.evaluate(
    val_ds,
    verbose=1
)

test_loss_m3, test_acc_m3 = best_mitigation_3.evaluate(
    test_ds,
    verbose=1
)

print("\nMitigation experiment 3")
print(f"Validation accuracy : {val_acc_m3:.4f} ({val_acc_m3*100:.2f}%)")
print(f"Validation loss     : {val_loss_m3:.4f}")
print(f"Test accuracy       : {test_acc_m3:.4f} ({test_acc_m3*100:.2f}%)")
print(f"Test loss           : {test_loss_m3:.4f}")

✓ Best Mitigation 3 checkpoint loaded
54/54 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.2849 - loss: 2.0321
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.3204 - loss: 2.0214

Mitigation experiment 3
Validation accuracy : 0.2849 (28.49%)
Validation loss     : 2.0321
Test accuracy       : 0.3204 (32.04%)
Test loss           : 2.0214


In [80]:
# ============================================================
# Step 11 — Group-level evaluation of Mitigation 3
# ============================================================

# Predictions on validation set
m3_val_probs = best_mitigation_3.predict(
    val_ds,
    verbose=1
)

m3_val_pred = np.argmax(m3_val_probs, axis=1)

# True labels in the SAME order as val_ds
m3_val_true = np.concatenate([
    y.numpy() for _, y in val_ds
])

print("Predictions:", len(m3_val_pred))
print("Labels     :", len(m3_val_true))
print(
    "Accuracy   :",
    np.mean(m3_val_pred == m3_val_true)
)

54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 111ms/step
Predictions: 1699
Labels     : 1699
Accuracy   : 0.28487345497351385


In [81]:
# Copy validation metadata
m3_val_results = val_df.reset_index(drop=True).copy()

m3_val_results["y_true"] = m3_val_true
m3_val_results["y_pred"] = m3_val_pred
m3_val_results["correct"] = (
    m3_val_results["y_true"] == m3_val_results["y_pred"]
).astype(int)

# Accuracy per validation group
mitigation_3_group_results = (
    m3_val_results
    .groupby(
        ["pathology", "split_group"],
        as_index=False
    )
    .agg(
        n_images=("correct", "size"),
        accuracy=("correct", "mean")
    )
)

mitigation_3_group_results["accuracy_pct"] = (
    100 * mitigation_3_group_results["accuracy"]
)

# Summary
m3_group_summary = pd.Series({
    "image_accuracy":
        m3_val_results["correct"].mean(),

    "macro_group_accuracy":
        mitigation_3_group_results["accuracy"].mean(),

    "median_group_accuracy":
        mitigation_3_group_results["accuracy"].median(),

    "zero_accuracy_groups":
        (mitigation_3_group_results["accuracy"] == 0).sum(),

    "n_groups":
        len(mitigation_3_group_results)
})

display(m3_group_summary)

,0
image_accuracy,0.284873
macro_group_accuracy,0.270569
median_group_accuracy,0.111111
zero_accuracy_groups,21.000000
n_groups,55.000000


In [82]:
# ============================================================
# Step 12 — Baseline vs Mitigation 3 by validation group
# ============================================================

comparison_m3 = (
    baseline_group_results[
        ["pathology", "split_group", "n_images", "accuracy"]
    ]
    .rename(columns={"accuracy": "baseline_accuracy"})
    .merge(
        mitigation_3_group_results[
            ["pathology", "split_group", "accuracy"]
        ].rename(columns={"accuracy": "m3_accuracy"}),
        on=["pathology", "split_group"],
        how="inner"
    )
)

comparison_m3["baseline_accuracy_pct"] = (
    100 * comparison_m3["baseline_accuracy"]
)

comparison_m3["m3_accuracy_pct"] = (
    100 * comparison_m3["m3_accuracy"]
)

comparison_m3["delta_pct"] = (
    comparison_m3["m3_accuracy_pct"]
    - comparison_m3["baseline_accuracy_pct"]
)

display(
    comparison_m3.sort_values("delta_pct")
)

print(
    "Groups improved:",
    (comparison_m3["delta_pct"] > 0).sum()
)

print(
    "Groups unchanged:",
    (comparison_m3["delta_pct"] == 0).sum()
)

print(
    "Groups degraded:",
    (comparison_m3["delta_pct"] < 0).sum()
)

print(
    "\nBaseline zero groups:",
    (comparison_m3["baseline_accuracy"] == 0).sum()
)

print(
    "M3 zero groups:",
    (comparison_m3["m3_accuracy"] == 0).sum()
)

print(
    "\nZero-accuracy groups rescued:",
    (
        (comparison_m3["baseline_accuracy"] == 0)
        & (comparison_m3["m3_accuracy"] > 0)
    ).sum()
)

print(
    "Groups newly collapsed to zero:",
    (
        (comparison_m3["baseline_accuracy"] > 0)
        & (comparison_m3["m3_accuracy"] == 0)
    ).sum()
)

,pathology,split_group,n_images,baseline_accuracy,m3_accuracy,baseline_accuracy_pct,m3_accuracy_pct,delta_pct
10,Ependymoma,60,50,0.240000,0.020000,24.000000,2.000000,-22.000000
11,Ependymoma,61,93,0.602151,0.387097,60.215054,38.709677,-21.505376
31,Meningioma,156,99,0.161616,0.000000,16.161616,0.000000,-16.161616
28,Meningioma,151,13,0.769231,0.615385,76.923077,61.538462,-15.384615
9,Ependymoma,57,13,0.384615,0.230769,38.461538,23.076923,-15.384615
29,Meningioma,153,37,0.459459,0.324324,45.945946,32.432432,-13.513514
13,Glioma,82,30,0.366667,0.233333,36.666667,23.333333,-13.333333
30,Meningioma,154,30,0.900000,0.766667,90.000000,76.666667,-13.333333
22,Meningioma,142,18,1.000000,0.888889,100.000000,88.888889,-11.111111
27,Meningioma,150,29,0.586207,0.482759,58.620690,48.275862,-10.344828


Groups improved: 12
Groups unchanged: 30
Groups degraded: 13

Baseline zero groups: 24
M3 zero groups: 21

Zero-accuracy groups rescued: 4
Groups newly collapsed to zero: 1


In [84]:
# ============================================================
# Step 13 — Class-level error diagnosis
# Baseline vs Mitigation 3
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# M3 classification report
# ------------------------------------------------------------

print("MITIGATION 3 — CLASSIFICATION REPORT\n")

print(
    classification_report(
        m3_val_true,
        m3_val_pred,
        target_names=CLASS_NAMES,
        digits=3,
        zero_division=0
    )
)

cm_m3 = confusion_matrix(
    m3_val_true,
    m3_val_pred
)

cm_m3_df = pd.DataFrame(
    cm_m3,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\nMITIGATION 3 — CONFUSION MATRIX")
display(cm_m3_df)

# ------------------------------------------------------------
# Normalized by TRUE class
# Each row sums to ~100%
# ------------------------------------------------------------

cm_m3_norm = (
    cm_m3_df
    .div(cm_m3_df.sum(axis=1), axis=0)
    * 100
)

print("\nMITIGATION 3 — NORMALIZED CONFUSION MATRIX (%)")
display(cm_m3_norm.round(1))

MITIGATION 3 — CLASSIFICATION REPORT

                    precision    recall  f1-score   support

       Astrocytoma      0.041     0.012     0.018       168
        Ependymoma      0.370     0.256     0.303       156
            Glioma      0.311     0.226     0.262       221
Hemangiopericytoma      0.000     0.000     0.000        89
        Meningioma      0.260     0.460     0.333       311
       Neurocytoma      1.000     0.031     0.060        97
            Normal      0.428     0.500     0.461       160
 Oligodendroglioma      0.370     0.207     0.266        82
             Other      0.123     0.113     0.118       230
        Schwannoma      0.322     0.665     0.434       185

          accuracy                          0.285      1699
         macro avg      0.322     0.247     0.225      1699
      weighted avg      0.293     0.285     0.247      1699


MITIGATION 3 — CONFUSION MATRIX


,Astrocytoma,Ependymoma,Glioma,Hemangiopericytoma,Meningioma,Neurocytoma,Normal,Oligodendroglioma,Other,Schwannoma
Astrocytoma,2,1,5,0,139,0,3,9,9,0
Ependymoma,9,40,43,0,1,0,8,0,1,54
Glioma,22,45,50,0,58,0,3,0,16,27
Hemangiopericytoma,1,0,2,0,72,0,7,0,7,0
Meningioma,0,1,12,0,143,0,2,11,129,13
Neurocytoma,1,11,16,0,2,3,9,0,12,43
Normal,7,7,9,2,23,0,80,5,7,20
Oligodendroglioma,2,0,0,0,38,0,22,17,3,0
Other,3,0,21,0,57,0,17,4,26,102
Schwannoma,2,3,3,0,16,0,36,0,2,123



MITIGATION 3 — NORMALIZED CONFUSION MATRIX (%)


,Astrocytoma,Ependymoma,Glioma,Hemangiopericytoma,Meningioma,Neurocytoma,Normal,Oligodendroglioma,Other,Schwannoma
Astrocytoma,1.2,0.6,3.0,0.0,82.7,0.0,1.8,5.4,5.4,0.0
Ependymoma,5.8,25.6,27.6,0.0,0.6,0.0,5.1,0.0,0.6,34.6
Glioma,10.0,20.4,22.6,0.0,26.2,0.0,1.4,0.0,7.2,12.2
Hemangiopericytoma,1.1,0.0,2.2,0.0,80.9,0.0,7.9,0.0,7.9,0.0
Meningioma,0.0,0.3,3.9,0.0,46.0,0.0,0.6,3.5,41.5,4.2
Neurocytoma,1.0,11.3,16.5,0.0,2.1,3.1,9.3,0.0,12.4,44.3
Normal,4.4,4.4,5.6,1.2,14.4,0.0,50.0,3.1,4.4,12.5
Oligodendroglioma,2.4,0.0,0.0,0.0,46.3,0.0,26.8,20.7,3.7,0.0
Other,1.3,0.0,9.1,0.0,24.8,0.0,7.4,1.7,11.3,44.3
Schwannoma,1.1,1.6,1.6,0.0,8.6,0.0,19.5,0.0,1.1,66.5


In [85]:
# ============================================================
# Step 14 — Class-level generalization diagnosis
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

cm_norm = confusion_matrix(
    m3_val_true,
    m3_val_pred,
    normalize="true"
)

rows = []

for i, true_class in enumerate(CLASS_NAMES):

    # Most common WRONG prediction
    wrong = cm_norm[i].copy()
    wrong[i] = -1

    j = np.argmax(wrong)

    train_groups = train_df.loc[
        train_df["pathology"] == true_class,
        "split_group"
    ].nunique()

    val_groups = val_df.loc[
        val_df["pathology"] == true_class,
        "split_group"
    ].nunique()

    train_images = (
        train_df["pathology"] == true_class
    ).sum()

    val_images = (
        val_df["pathology"] == true_class
    ).sum()

    rows.append({
        "pathology": true_class,
        "train_images": train_images,
        "train_groups": train_groups,
        "val_images": val_images,
        "val_groups": val_groups,
        "m3_recall_pct": 100 * cm_norm[i, i],
        "main_confusion": CLASS_NAMES[j],
        "main_confusion_pct": 100 * cm_norm[i, j]
    })

class_generalization = pd.DataFrame(rows)

display(
    class_generalization.sort_values("m3_recall_pct")
)

,pathology,train_images,train_groups,val_images,val_groups,m3_recall_pct,main_confusion,main_confusion_pct
3,Hemangiopericytoma,428,7,89,2,0.000000,Meningioma,80.898876
0,Astrocytoma,782,39,168,9,1.190476,Meningioma,82.738095
5,Neurocytoma,436,11,97,2,3.092784,Schwannoma,44.329897
8,Other,1073,39,230,9,11.304348,Schwannoma,44.347826
7,Oligodendroglioma,385,16,82,4,20.731707,Meningioma,46.341463
2,Glioma,1034,35,221,8,22.624434,Meningioma,26.244344
1,Ependymoma,725,17,156,3,25.641026,Schwannoma,34.615385
4,Meningioma,1449,45,311,10,45.980707,Other,41.479100
6,Normal,750,10,160,2,50.000000,Meningioma,14.375000
9,Schwannoma,866,28,185,6,66.486486,Normal,19.459459


In [86]:
# ============================================================
# Step 15 — Group-level confusion routes
# ============================================================

val_confusion_df = val_df.reset_index(drop=True).copy()

assert len(val_confusion_df) == len(m3_val_true)
assert len(val_confusion_df) == len(m3_val_pred)

val_confusion_df["true_idx"] = m3_val_true
val_confusion_df["pred_idx"] = m3_val_pred

val_confusion_df["true_class"] = [
    CLASS_NAMES[i] for i in m3_val_true
]

val_confusion_df["pred_class"] = [
    CLASS_NAMES[i] for i in m3_val_pred
]

# ------------------------------------------------------------
# For every validation group:
# dominant prediction + its proportion
# ------------------------------------------------------------

group_routes = (
    val_confusion_df
    .groupby(
        ["pathology", "split_group"],
        as_index=False
    )
    .agg(
        n_images=("pred_class", "size"),
        correct=("pred_idx",
                 lambda x: np.nan)  # replaced below
    )
)

# Accuracy separately
group_acc = (
    val_confusion_df
    .assign(
        correct=lambda x:
        x["true_idx"] == x["pred_idx"]
    )
    .groupby(
        ["pathology", "split_group"]
    )["correct"]
    .mean()
    .reset_index(name="accuracy")
)

# Dominant predicted class
dominant = (
    val_confusion_df
    .groupby(
        ["pathology", "split_group"]
    )["pred_class"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
    .reset_index(name="dominant_prediction")
)

# Percentage assigned to dominant prediction
dominant_pct = (
    val_confusion_df
    .groupby(
        ["pathology", "split_group"]
    )["pred_class"]
    .agg(
        lambda x:
        100 * x.value_counts().iloc[0] / len(x)
    )
    .reset_index(name="dominant_prediction_pct")
)

group_routes = (
    val_confusion_df
    .groupby(
        ["pathology", "split_group"]
    )
    .size()
    .reset_index(name="n_images")
    .merge(group_acc,
           on=["pathology", "split_group"])
    .merge(dominant,
           on=["pathology", "split_group"])
    .merge(dominant_pct,
           on=["pathology", "split_group"])
)

group_routes["accuracy_pct"] = (
    100 * group_routes["accuracy"]
)

group_routes = group_routes.sort_values(
    ["pathology", "accuracy_pct"]
)

display(group_routes)

,pathology,split_group,n_images,accuracy,dominant_prediction,dominant_prediction_pct,accuracy_pct
0,Astrocytoma,0,11,0.000000,Glioma,36.363636,0.000000
1,Astrocytoma,2,6,0.000000,Meningioma,83.333333,0.000000
3,Astrocytoma,4,47,0.000000,Meningioma,97.872340,0.000000
4,Astrocytoma,5,3,0.000000,Meningioma,100.000000,0.000000
5,Astrocytoma,6,4,0.000000,Oligodendroglioma,100.000000,0.000000
7,Astrocytoma,8,3,0.000000,Meningioma,33.333333,0.000000
8,Astrocytoma,10,12,0.000000,Meningioma,91.666667,0.000000
2,Astrocytoma,3,53,0.018868,Meningioma,86.792453,1.886792
6,Astrocytoma,7,29,0.034483,Meningioma,82.758621,3.448276
10,Ependymoma,60,50,0.020000,Glioma,58.000000,2.000000
